# Chapter 6 Sample Code
This file produces the letter frequencies, ngrams, and uses them to produce sequences of letters using a statistical approach.

In [1]:
import pandas as pd
import os
import re
import pickle
import random

### Download the Reuters dataset 

The next few files parse the Reuters dataset files available here:
https://kdd.ics.uci.edu/databases/reuters21578/reuters21578.html

The files should be unzipped and untarred into the directory listed below with many .sgm files.

The frequency data are then written to .pkl files.

In [2]:
directory = 'ch6/'

### Parsing the Reuters sgm files into pkl frequency files

In [3]:
def extractBodyTextFromFile(filename):
    '''Extracts text from a filename, looking for <BODY> tags and returning the content between them.'''
    inBody = False
    textblock = []
    with open(filename, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.readline()
        while len(text) > 0:
            if '<BODY>' in text and '</BODY>' in text:
                inBody = False
                text = re.sub('^.*\<BODY\>', '', text)
                text = re.sub('\<\/BODY\>.*$', '', text)
                textblock.append(text)
            elif '<BODY>' in text and inBody is False:
                text = re.sub('^.*\<BODY\>', '', text)
                textblock.append(text)
                inBody = True
            elif '</BODY>' in text and inBody is True:
                text = re.sub('\<\/BODY\>.*$', '', text)
                textblock.append(text)
                inBody = False
            elif inBody:
                textblock.append(text)
            text = f.readline()
    return textblock

<>:10: SyntaxWarning: invalid escape sequence '\<'
<>:11: SyntaxWarning: invalid escape sequence '\<'
<>:14: SyntaxWarning: invalid escape sequence '\<'
<>:18: SyntaxWarning: invalid escape sequence '\<'
<>:10: SyntaxWarning: invalid escape sequence '\<'
<>:11: SyntaxWarning: invalid escape sequence '\<'
<>:14: SyntaxWarning: invalid escape sequence '\<'
<>:18: SyntaxWarning: invalid escape sequence '\<'
C:\Users\nguye\AppData\Local\Temp\ipykernel_24068\183685533.py:10: SyntaxWarning: invalid escape sequence '\<'
  text = re.sub('^.*\<BODY\>', '', text)
C:\Users\nguye\AppData\Local\Temp\ipykernel_24068\183685533.py:11: SyntaxWarning: invalid escape sequence '\<'
  text = re.sub('\<\/BODY\>.*$', '', text)
C:\Users\nguye\AppData\Local\Temp\ipykernel_24068\183685533.py:14: SyntaxWarning: invalid escape sequence '\<'
  text = re.sub('^.*\<BODY\>', '', text)
C:\Users\nguye\AppData\Local\Temp\ipykernel_24068\183685533.py:18: SyntaxWarning: invalid escape sequence '\<'
  text = re.sub('\<\/BO

In [4]:
def preprocess(text_block):
    """Normalize whitespace while keeping punctuation and original casing.

    Chapter 6 later explores why keeping punctuation and capitalization can help
    a character-level model learn sentence-like structure.
    """
    text = " ".join(text_block)
    text = re.sub(r"[\s\r\n]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


In [5]:
def computeFrequencies(text, ngram_len, frequencies):
    '''Updates and returns frequencies dictionary with ngrams of ngram_len from text.'''
    for i in range(len(text)-ngram_len):
        key = text[i:i+ngram_len]
        if key in frequencies:
            frequencies[key] += 1 
        else:
            frequencies[key] = 1
    return frequencies

In [6]:
def purge_nonletters(ngrams):
    """Remove n-grams containing characters outside our allowed set.

    Allowed characters:
    - letters (a-z, A-Z)
    - whitespace
    - period '.' and comma ','

    This is used for character-level n-grams.
    """
    allowed = re.compile(r"^[A-Za-z\s\.,]+$")
    to_delete = []
    for key in ngrams:
        if allowed.match(key):
            continue
        to_delete.append(key)
    for rejected in to_delete:
        ngrams.pop(rejected, None)
    return ngrams


## 3) Tinh chỉnh dữ liệu: giữ lại dấu câu và chữ hoa

Trong bản gốc, dữ liệu bị ép `lower()` và các n-gram có ký tự “lạ” bị loại bỏ, làm mất cấu trúc tự nhiên của câu.

Mình đã sửa:
- `preprocess()` không còn ép chữ thường (giữ chữ hoa).
- `purge_nonletters()` cho phép giữ dấu chấm `.` và dấu phẩy `,`.

Gợi ý thử nghiệm: chạy lại phần tạo `freq5.pkl`/`freq6.pkl`, rồi đặt `ngram_len = 5` hoặc `6` ở phần cuối để sinh văn bản. Bạn sẽ thường thấy sau dấu `.` là khoảng trắng + chữ hoa.

In [7]:
def computeFreq(directory, ngram_len):
    '''Primary routine to iterate through all .sigm files in directory and compute ngram frequencies.'''
    sgm_files = [f for f in os.listdir(directory) if f.endswith('.sgm')]
    ngram_freq = {}
    for file in sgm_files:
        extract_text_blocks = extractBodyTextFromFile(os.path.join(directory, file))
        cleaned_block = preprocess(extract_text_blocks)
        ngram_freq = computeFrequencies(cleaned_block, ngram_len, ngram_freq)
        purge_nonletters(ngram_freq)
    return ngram_freq

## 1) Word-level n-grams (từ vựng) và “bùng nổ tổ hợp”

Ở cấp độ chữ cái, số tổ hợp n-gram còn tương đối nhỏ. Khi chuyển sang cấp độ từ (word-level), số n-gram khác nhau tăng cực nhanh theo $n$ (và theo kích thước từ vựng), nên với cùng tập dữ liệu Reuters bạn sẽ thấy `dictionary size` “bùng nổ” rất rõ.

Bên dưới là phiên bản *copy* của `computeFrequencies` nhưng chạy trên danh sách từ thay vì lát chuỗi theo ký tự.

In [8]:
def preprocess_words(text_block):
    """Turn a Reuters BODY text block into a list of word tokens.

    Uses `preprocess()` for whitespace normalization, then extracts word tokens.
    (For Word2Vec and word-level n-grams we typically normalize to lowercase.)
    """
    text = preprocess(text_block)
    tokens = re.findall(r"[A-Za-z]+", text)
    return [t.lower() for t in tokens if t]


def computeWordFrequencies(words, ngram_len, frequencies):
    """Updates and returns frequencies dict with word n-grams from a word list."""
    for i in range(len(words) - ngram_len + 1):
        key = tuple(words[i : i + ngram_len])
        frequencies[key] = frequencies.get(key, 0) + 1
    return frequencies


def computeWordFreq(directory, ngram_len, max_files=None):
    """Compute word-level n-gram frequencies for Reuters .sgm files."""
    sgm_files = [f for f in os.listdir(directory) if f.endswith('.sgm')]
    if max_files is not None:
        sgm_files = sgm_files[:max_files]

    ngram_freq = {}
    for file in sgm_files:
        extract_text_blocks = extractBodyTextFromFile(os.path.join(directory, file))
        words = preprocess_words(extract_text_blocks)
        ngram_freq = computeWordFrequencies(words, ngram_len, ngram_freq)
    return ngram_freq


# Demo: print dictionary size growth (set max_files to keep runtime reasonable)
MAX_WORD_NGRAM = 5
MAX_FILES = 10  # increase to see the combinatorial explosion more strongly
for ngrami in range(1, MAX_WORD_NGRAM + 1):
    print('Word-level ngram size', ngrami)
    freqOutput = computeWordFreq(directory, ngrami, max_files=MAX_FILES)
    print('  Dictionary size:', len(freqOutput))


Word-level ngram size 1
  Dictionary size: 27305
Word-level ngram size 2
  Dictionary size: 331577
Word-level ngram size 3
  Dictionary size: 729698
Word-level ngram size 4
  Dictionary size: 944656
Word-level ngram size 5
  Dictionary size: 1037520


## 2) Huấn luyện Word2Vec (Gensim)

Word2Vec học embedding để “gom” các từ có ngữ nghĩa gần nhau vào gần nhau trong không gian vector. Với Reuters, bạn có thể thử `most_similar('bank')` hoặc `most_similar('finance')` để quan sát.

Yêu cầu: cài `gensim` (đã được thêm vào `requirements.txt`).

In [9]:
# If you haven't installed dependencies yet, run (in a terminal):
#   pip install -r requirements.txt

from gensim.models import Word2Vec


def build_reuters_corpus(directory, max_files=None):
    """Build a list-of-lists corpus for Word2Vec training."""
    sgm_files = [f for f in os.listdir(directory) if f.endswith('.sgm')]
    if max_files is not None:
        sgm_files = sgm_files[:max_files]

    sentences = []
    for file in sgm_files:
        extract_text_blocks = extractBodyTextFromFile(os.path.join(directory, file))
        tokens = preprocess_words(extract_text_blocks)
        if tokens:
            sentences.append(tokens)
    return sentences


# Train on a subset first (increase MAX_FILES for better quality)
MAX_FILES = 50
sentences = build_reuters_corpus(directory, max_files=MAX_FILES)
print('Num training documents:', len(sentences))

model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=(os.cpu_count() or 1),
    sg=1,  # skip-gram
    epochs=10,
)

for query in ['bank', 'finance']:
    try:
        print('\nMost similar to:', query)
        print(model.wv.most_similar(query, topn=10))
    except KeyError:
        print(f"\n'{query}' not in vocabulary (try increasing MAX_FILES or lowering min_count).")


Num training documents: 22

Most similar to: bank
[('central', 0.6519407629966736), ('governor', 0.6442280411720276), ('lloyds', 0.6437005400657654), ('bankers', 0.6285724639892578), ('banks', 0.620811939239502), ('fhlbb', 0.5924832820892334), ('westminster', 0.591015100479126), ('england', 0.5906015634536743), ('paper', 0.5898746252059937), ('fdic', 0.5889578461647034)]

Most similar to: finance
[('balladur', 0.7228733897209167), ('edouard', 0.7140117883682251), ('prime', 0.685403048992157), ('belgian', 0.6779603362083435), ('planning', 0.6748579144477844), ('gerhard', 0.6714532971382141), ('italian', 0.6696444749832153), ('miyazawa', 0.6676364541053772), ('michael', 0.6660297513008118), ('affairs', 0.6646235585212708)]


Computes ngram frequencies for 1 to MAX_NGRAM_SIZE and write to freq#.pkl.

In [10]:
MAX_NGRAM_SIZE = 7
for ngrami in range(1,MAX_NGRAM_SIZE+1):
    print('Processing ngram size', ngrami)
    freqOutput = computeFreq(directory, ngrami)
    with open(os.path.join(directory, 'freq' + str(ngrami) + '.pkl'), 'wb') as f:
        pickle.dump(freqOutput, f)
    print('Length of freq dictionary size', ngrami, len(freqOutput))    

Processing ngram size 1
Length of freq dictionary size 1 55
Processing ngram size 2
Length of freq dictionary size 2 2097
Processing ngram size 3
Length of freq dictionary size 3 20964
Processing ngram size 4
Length of freq dictionary size 4 94813
Processing ngram size 5
Length of freq dictionary size 5 283626
Processing ngram size 6
Length of freq dictionary size 6 649600
Processing ngram size 7
Length of freq dictionary size 7 1199657


In [11]:
def checkTrainingSetSize(directory):
    ''' Checks the size of the training set before and after removing non-letter characters. '''
    sgm_files = [f for f in os.listdir(directory) if f.endswith('.sgm')]
    training_set_size = 0
    cleaned_set_size = 0
    for file in sgm_files:
        extract_text_blocks = extractBodyTextFromFile(os.path.join(directory, file))
        training_set_size += len(' '.join(extract_text_blocks))
        cleaned_block = preprocess(extract_text_blocks)
        cleaned_set_size += len(cleaned_block)
    print('Uncleaned', training_set_size, 'Cleaned', cleaned_set_size)
checkTrainingSetSize(directory)


Uncleaned 16372459 Cleaned 15620895


Generate frequency dict of the letters within the text files.
Write the file to letter_frequencies.csv

In [12]:
def computeIndividualLetterFrequencies(directory):
    '''Computes letter frequencies in the text files in the given directory.'''
    sgm_files = [f for f in os.listdir(directory) if f.endswith('.sgm')]
    training_set_size = 0
    cleaned_set_size = 0
    letter_frequencies = {}
    for u in 'abcdefghijklmnopqrstuvwxyz':
        letter_frequencies[u] = 0
    for file in sgm_files:
        extract_text_blocks = extractBodyTextFromFile(os.path.join(directory, file))
        training_set_size += len(' '.join(extract_text_blocks))
        cleaned_block = preprocess(extract_text_blocks)
        for ch in cleaned_block:
            chl = ch.lower()
            if chl in letter_frequencies:
                letter_frequencies[chl] += 1
        cleaned_set_size += len(cleaned_block)
    return letter_frequencies

letter_frequencies = computeIndividualLetterFrequencies(directory)
df = pd.DataFrame(list(letter_frequencies.items()), columns=['letter', 'frequency'])
df = df.sort_values(by='frequency', ascending=False)
df.to_csv('letter_frequencies.csv', index=False)


### Generate the next letters from the ngram frequency data

In [13]:
def create_next_letter_from_current_frequency_table():
    ''' Generates pairwise frequency table for current letter and next letter'''
    with open(os.path.join(directory, 'freq2.pkl'), 'rb') as f:
        freq2 = pickle.load(f)
    set1 = set()
    set2 = set()
    for key in freq2:
        set1.add(key[0])
        set2.add(key[1])
    lst1 = list(set1)
    lst2 = list(set2)
    lst1.sort()
    lst2.sort()
    freqTable = pd.DataFrame(index=lst2, columns=lst1)    
    freqTable.fillna(0, inplace=True)
    for key in freq2:
        freqTable.at[key[1], key[0]] = freq2[key]
    freqTable.to_csv(os.path.join(directory, 'freq2tbl.csv'))
create_next_letter_from_current_frequency_table()

C:\Users\nguye\AppData\Local\Temp\ipykernel_24068\2664634740.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  freqTable.fillna(0, inplace=True)


The file written out was freq2tbl was used to create the histogram of the most frequent single entries and the pairs of entries.

In [14]:
def loadNormalizePkl1(filename):
    with open(filename, 'rb') as f:
        data = pickle.load(f)
    totalsum = 0.0
    for key, val in data.items():
        totalsum += val
    for key in data:
        data[key] = data[key] / totalsum
    return data



Writes text using the letter frequencies alone without ngrams.

In [15]:
data = loadNormalizePkl1(os.path.join(directory,'freq1.pkl'))
random.seed(41)
for iter in range(10):
    outputstr = ''
    for index in range(70):
        prob = random.uniform(0.0, 1.0)
        sum = 0
        done = False
        prevkey = ''
        for key in data:
            if not done:
                sum += data[key]
                if sum >= prob:
                    done = True
                    outputstr += key

    print(iter+1, outputstr)

1  reptun dtail eedeitoaollrddeiepuIo tteJl ei ic he d gnyi eoolvgeiemig
2 t l.r di  .srusm p ygdreoeiorte rieEtdncTmahtfiniedaont ipm elt,vst au
3 rntrecocf,o,ddhara  elrvneinhashad entuctdeeckal de olKik ao  leewl m 
4 lrmAyump.sla em nycsunsumraknrisslswlP  nkhirf b iors e wt.ltqdi ommau
5 a.Aahbt  sottopmln   CpmnnuseNsscwne,oasaaCeNe hconfaiwj besbge nsron 
6  c  aew sclbapr essr fnmngitptnre  ebh s l. dms  l meiio l o n  fi  ar
7 caweoiaito   s roth  ad hddiDc neieituedbYaeaeIoh obfsv aef r SoctEn s
8 hpuskui vfBe ibncoosirkisnloig. ia agb enipknlltshth Rsud tyd Piraayla
9  kaarjtpnatscaspdandi Stun neioeobpifricetesermcneeeutgts  fu,i.aSoomr
10  trsye, u iu s ghr unrytm bt eaeaaeduwBb n rei.ehnsvsh    s.hrarr izae


#### Produce text using the n-gram letter frequencies of different lengths.

In [16]:
def loadNormalizePkln(filename):
    ''' Creates normalization by first n-1 letters of ngram for the last letter in ngrams'''
    with open(filename, 'rb') as f:
        data = pickle.load(f)
    norm = {}
    for key in data.keys():
        prefix = key[0:len(key)-1]
        norm[prefix] = 0
    for key,value in data.items():
        prefix = key[0:len(key)-1]
        norm[prefix] += value
    for key,value in data.items():
        prefix = key[0:len(key)-1]
        data[key] = data[key] / norm[prefix]
    return data

In [17]:
def find_prob_string(prob, data, ngram):
    '''Selects random entry using prob from the data keys starting with ngram that have been normalized.'''
    cumulative_prob = 0.0
    for key in data:
        if key.startswith(ngram):
            cumulative_prob += data[key]
            if cumulative_prob >= prob:
                return key[-1]
    print('No match found for', ngram, prob)
    return None


def find_max_prob_string(data, ngram):
    '''Greedy choice: always pick the next character with the highest probability.'''
    best_key = None
    best_prob = -1.0
    for key, prob in data.items():
        if key.startswith(ngram) and prob > best_prob:
            best_prob = prob
            best_key = key

    if best_key is None:
        print('No match found for', ngram)
        return None
    return best_key[-1]


In [18]:
def return_random_key(data):
    ''' Returns a random key from the data dictionary.'''
    key = random.choice(list(data.keys()))  
    return key
    

In [19]:
def createNgramStr(directory, ngram, strategy='random'):
    '''Generates lines of random text based on n-gram frequencies stored in directory.

    strategy:
      - 'random': sample next char based on probabilities (like temperature > 0)
      - 'greedy': always pick the max-probability next char (often degenerates)
    '''
    NUM_LINES = 5
    LETTERS_PER_LINE = 70
    data = loadNormalizePkln(os.path.join(directory, 'freq' + str(ngram) + '.pkl'))
    random.seed(422)

    for iter in range(NUM_LINES):
        outputstr = return_random_key(data)
        for index in range(LETTERS_PER_LINE):
            context = outputstr[-(ngram - 1) :] if ngram > 1 else ''

            if strategy == 'greedy':
                nextchar = find_max_prob_string(data, context)
            else:
                prob = random.uniform(0.0, 1.0)
                nextchar = find_prob_string(prob, data, context)

            if nextchar is None:
                break
            outputstr += nextchar
        print(iter + 1, outputstr)


##### Main call to generate ngram-based letter sequences using probabilities

## 4) Greedy Search vs Ngẫu nhiên (Random)

- Random sampling giúp văn bản đa dạng hơn.
- Greedy (luôn chọn xác suất lớn nhất) thường bị “kẹt” vào vòng lặp (degenerate loop).

Đây là lý do các hệ thống như ChatGPT/Copilot cần tham số kiểu *temperature* để điều khiển độ ngẫu nhiên.

In [20]:
ngram_len = 6  # try 5 or 6 to see sentence-like structure with punctuation + capitalization

print('--- Random sampling ---')
createNgramStr(directory, ngram_len, strategy='random')

print('\n--- Greedy (max probability) ---')
createNgramStr(directory, ngram_len, strategy='greedy')


--- Random sampling ---
1 mb...but its speculate. The unit, has included a letter relations nonpartici
2 ra Ecuador owes Inc. The Banking is doing the was up on agreement stressed o
3 ng WTC Internating to acquisitions, announce, policy involved by more that t
4 ectady, pegged slump in Mouse, I would debentures due to around extend the l
5 y particular rose for to three of its for the plan to with the values of yea

--- Greedy (max probability) ---
1 mb...but it will be a statement of the company said it will be a statement o
2 le Ecuador workers said it will be a statement of the company said it will b
3 ng that the company said it will be a statement of the company said it will 
4 gh Israel and the company said it will be a statement of the company said it
5 o, dealers said it will be a statement of the company said it will be a stat
